In [ ]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import urllib.request
import tqdm

# 모델과 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained('skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', pad_token='<pad>')
model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)


In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatBotData.csv")
train_data = pd.read_csv('ChatBotData.csv')
print('챗봇 데이터의 개수 :', len(train_data))

챗봇 데이터의 개수 : 11823


In [ ]:
class ChatbotDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        question = self.data.iloc[idx]['Q']
        answer = self.data.iloc[idx]['A']
        bos_token = [self.tokenizer.bos_token_id]
        eos_token = [self.tokenizer.eos_token_id]
        input_ids = bos_token + self.tokenizer.encode('<usr>' + question) + self.tokenizer.encode('<sys>' + answer) + eos_token
        input_ids = input_ids[:self.max_len]  # 최대 길이 제한
        return torch.tensor(input_ids)

In [ ]:
def collate_fn(batch):
    input_ids = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return input_ids, attention_mask

batch_size = 16
dataset = ChatbotDataset(train_data, tokenizer)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

In [ ]:
# 옵티마이저 정의
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

accumulation_steps = 4  # 메모리 사용량이 줄어듦

<ipython-input-6-75e9ff379d41>:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# 학습 루프
model.train()
EPOCHS = 15

for epoch in range(EPOCHS):
    epoch_loss = 0
    for step, batch in enumerate(tqdm.tqdm(dataloader)):
        input_ids, attention_mask = [x.to(device) for x in batch]
        labels = input_ids.clone().detach()

        # 모델의 forward pass (Mixed Precision 사용)
        optimizer.zero_grad()
        with autocast():  # Mixed Precision
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / accumulation_steps  # Gradient Accumulation

        # backward pass (Mixed Precision 및 Gradient Accumulation 사용)
        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / len(dataloader)
    print(f'[Epoch: {epoch + 1}] Loss = {avg_epoch_loss:.9f}')

  0%|          | 0/739 [00:00<?, ?it/s]<ipython-input-7-e2f369b4919b>:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # Mixed Precision
100%|██████████| 739/739 [00:53<00:00, 13.92it/s]


[Epoch: 1] Loss = 0.621132263


100%|██████████| 739/739 [00:50<00:00, 14.51it/s]


[Epoch: 2] Loss = 0.509283366


100%|██████████| 739/739 [00:50<00:00, 14.54it/s]


...

[Epoch: 13] Loss = 0.312435804


100%|██████████| 739/739 [00:51<00:00, 14.44it/s]


[Epoch: 14] Loss = 0.301219914


100%|██████████| 739/739 [00:51<00:00, 14.44it/s]

[Epoch: 15] Loss = 0.290862420


In [ ]:
def generate_text(input_text, model, tokenizer, max_length=150, top_k=50, temperature=0.9):
    model.eval()
    with torch.no_grad():
        input_ids = tokenizer.encode('<usr>' + input_text + '<sys>', return_tensors='pt').to(device)
        output = model.generate(input_ids,
                                max_length=max_length,
                                do_sample=True,
                                top_k=top_k,
                                temperature=temperature,
                                eos_token_id=tokenizer.eos_token_id)
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        response = decoded_output.split('<sys> ')[1] if '<sys> ' in decoded_output else decoded_output
        return response

In [ ]:
def return_answer_by_chatbot(user_text):
    chatbot_response = generate_text(user_text, model, tokenizer)
    if user_text in chatbot_response:
        chatbot_response = chatbot_response.replace(user_text, "").strip()

    print(f"입력: {user_text}")
    print(f"출력: {chatbot_response}\n")

In [ ]:
return_answer_by_chatbot('안녕! 반가워~')

입력: 안녕! 반가워~
출력: 안녕히 주무세요.



In [ ]:
return_answer_by_chatbot('너무 심심한데 나랑 놀자')

입력: 너무 심심한데 나랑 놀자
출력: 놀고 자고 일어나세요.



In [ ]:
return_answer_by_chatbot('영화 해리포터 재밌어?')

입력: 영화 해리포터 재밌어?
출력: 최신 영화 추천해드립니다.



In [ ]:
model.save_pretrained('qna_model/bert-base')
tokenizer.save_pretrained('qna_model/bert-base')

('qna_model/bert-base/tokenizer_config.json',
 'qna_model/bert-base/special_tokens_map.json',
 'qna_model/bert-base/vocab.json',
 'qna_model/bert-base/merges.txt',
 'qna_model/bert-base/added_tokens.json',
 'qna_model/bert-base/tokenizer.json')

# TTS 활용

In [ ]:
!pip install gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

# 저장된 모델과 토크나이저 불러오기
def load_model_and_tokenizer(model_dir='qna_model/bert-base'):
    model = GPT2LMHeadModel.from_pretrained(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(51200, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=51200, bias=False)
)

In [ ]:
# 텍스트를 음성 파일로 변환 및 Audio로 재생하는 함수
def text_to_speech_and_play(text, lang='ko', file_name="response.mp3"):
    tts = gTTS(text=text, lang=lang)
    tts.save(file_name)
    print(f"TTS 음성 파일이 '{file_name}'로 저장되었습니다.")

    return Audio(file_name, autoplay=True)

In [ ]:
# 텍스트 생성 함수 (GPT-2 모델을 사용)
def generate_text(input_text, model, tokenizer, max_length=100, top_k=50, temperature=0.7):
    model.eval()
    with torch.no_grad():
        input_ids = tokenizer.encode('<usr>' + input_text + '<sys>', return_tensors='pt').to(device)
        max_length = max_length if max_length is not None else len(input_ids[0]) + 50
        output = model.generate(input_ids,
                                max_length=max_length,
                                do_sample=True,
                                top_k=top_k,
                                temperature=temperature,
                                eos_token_id=tokenizer.eos_token_id)
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        response = decoded_output.split('<sys> ')[1] if '<sys> ' in decoded_output else decoded_output
        return response

In [ ]:
# 음성 챗봇 함수 (파일로 저장된 음성을 바로 재생)
def voice_chatbot(user_text):
    chatbot_response = generate_text(user_text, model, tokenizer)
    print(f"Bot: {chatbot_response}")
    return text_to_speech_and_play(chatbot_response)

In [ ]:
voice_chatbot("안녕하세요, 챗봇!")

Bot: 안녕하세요, 챗봇! 제가 응원할게요.
TTS 음성 파일이 'response.mp3'로 저장되었습니다.
